# Lab 3: Loop de Treino do Zero

## Lab 3: Loop de Treino do Zero

Usamos `sshleifer/tiny-gpt2` aqui (não o SmolLM2) — pra este lab, o que
importa é a **mecânica do treino** (loss caindo, gradientes fluindo), não a
qualidade do texto gerado. Um modelo minúsculo deixa o loop rápido o
suficiente pra rodar vários steps em segundos.

In [1]:
!pip install -q torch transformers matplotlib

import matplotlib
matplotlib.use("Agg")

import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "sshleifer/tiny-gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
print(f"✓ Modelo carregado: {sum(p.numel() for p in model.parameters()):,} parâmetros")

✓ Modelo carregado: 102,714 parâmetros


### 1. Preparando dados (input = label deslocado)

In [2]:
texts = [
    "The transformer architecture uses self-attention.",
    "Gradient descent optimizes the model weights.",
    "LoRA reduces the number of trainable parameters.",
]

def prepare_batch(texts):
    enc = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=32)
    labels = enc["input_ids"].clone()
    labels[enc["attention_mask"] == 0] = -100  # ignora padding no cálculo do loss
    return enc["input_ids"], enc["attention_mask"], labels

input_ids, attention_mask, labels = prepare_batch(texts)
print(f"input_ids shape: {input_ids.shape}")
print(f"Primeira sequência de tokens: {input_ids[0][:10].tolist()}")
print(f"Labels (deveria ser igual, deslocado internamente pelo modelo): {labels[0][:10].tolist()}")

input_ids shape: torch.Size([3, 10])
Primeira sequência de tokens: [464, 47385, 10959, 3544, 2116, 12, 1078, 1463, 13, 50256]
Labels (deveria ser igual, deslocado internamente pelo modelo): [464, 47385, 10959, 3544, 2116, 12, 1078, 1463, 13, -100]


**Resultado esperado:** `input_ids` e `labels` começam idênticos — o
deslocamento (prever o próximo token) acontece **dentro** do modelo (ele
compara `logits[:, i]` com `labels[:, i+1]` automaticamente), não
manualmente aqui.

### 2. Medindo memória: com e sem gradientes

**Por que importa (Semana 3.9):** confirma na prática que ativar o cálculo
de gradiente (preparar pra treinar) usa mais memória que só rodar o
modelo (inferência).

In [3]:
import gc

def measure_forward_memory(requires_grad: bool):
    gc.collect()
    model.zero_grad(set_to_none=True)
    for p in model.parameters():
        p.requires_grad_(requires_grad)

    with torch.set_grad_enabled(requires_grad):
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    n_tensors_with_grad = sum(1 for p in model.parameters() if p.grad is not None)
    return outputs.loss.item()

loss_no_grad = measure_forward_memory(requires_grad=False)
loss_with_grad = measure_forward_memory(requires_grad=True)
print(f"Loss (sem grad, só inferência): {loss_no_grad:.4f}")
print(f"Loss (com grad, pronto pra treinar): {loss_with_grad:.4f}")
print("(os valores de loss são iguais — a diferença é só se o grafo de gradiente foi construído)")

Loss (sem grad, só inferência): 10.8305
Loss (com grad, pronto pra treinar): 10.8305
(os valores de loss são iguais — a diferença é só se o grafo de gradiente foi construído)


**Resultado esperado:** os dois valores de loss são praticamente idênticos
(mesma forward pass matemática) — a diferença real é invisível aqui
(memória), mas fica clara no próximo passo quando só a versão
`requires_grad=True` permite `.backward()`.

### 3. O loop de treino de verdade

In [4]:
model.train()
for p in model.parameters():
    p.requires_grad_(True)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-3)
losses = []

n_steps = 50
for step in range(n_steps):
    optimizer.zero_grad()
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping (Semana 3.7)
    optimizer.step()

    losses.append(loss.item())
    if step % 5 == 0:
        print(f"Step {step}: loss = {loss.item():.4f}")

print(f"\n✓ Loss inicial: {losses[0]:.4f} → Loss final: {losses[-1]:.4f}")

Step 0: loss = 10.8321
Step 5: loss = 10.7898
Step 10: loss = 10.7000
Step 15: loss = 10.5915
Step 20: loss = 10.4670
Step 25: loss = 10.3263
Step 30: loss = 10.1688
Step 35: loss = 9.9935
Step 40: loss = 9.7993
Step 45: loss = 9.5856

✓ Loss inicial: 10.8321 → Loss final: 9.4001


**Resultado esperado:** o loss cai de forma clara e consistente, de
~10.83 pra ~9.50 em 50 steps (o modelo está literalmente decorando essas 3
frases — com um dataset tão pequeno, isso é esperado e demonstra que o
mecanismo de treino funciona, não que o modelo "aprendeu linguagem"). Se
o loss não caísse nada, seria sinal de bug no loop (LR zerado, gradientes
não conectados, etc.) — por isso essa checagem é útil mesmo num exemplo de
brinquedo.

### 4. Visualizando a curva de loss

In [5]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(losses, marker="o", markersize=3)
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title("Loss caindo ao longo do treino (tiny-gpt2, 3 frases, overfitting proposital)")
ax.grid(alpha=0.3)
plt.savefig("loss_curve.png", dpi=80, bbox_inches="tight")
plt.close(fig)
print("✓ Curva de loss salva em loss_curve.png")

✓ Curva de loss salva em loss_curve.png


**Resultado esperado:** uma curva descendente, provavelmente com alguns
solavancos (é um dataset de 3 frases só, sem mini-batching real) mas com
tendência clara de queda entre o início e o fim.

**Próximos passos:** Semana 4 aborda como preparar um dataset de verdade
(muito maior, limpo, formatado) antes de aplicar esse mesmo loop de treino
num cenário real — Semana 5 (SFT) usa exatamente essa mecânica, só que com
uma biblioteca de produção (`TRL`) em vez do loop manual.